# Course 9: Ten Years of Deep Learning Ideas
## Session 6: In-context learning and learning to learn

**Goal:** train prototypical networks and first-order MAML on few-shot tasks, then train a tiny transformer to do linear regression in context and compare it with least squares and one step of gradient descent.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Setup
Three experiments: few-shot classification with prototypical networks, fast adaptation with first-order MAML, and a tiny transformer that learns linear regression **in context**. Meta-learning papers use benchmarks such as Omniglot (1,623 handwritten characters) and miniImageNet; we build small stand-ins so that each part runs in seconds.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

torch.set_num_threads(2)          # tiny models run as fast on 2 threads, and shared machines stay responsive
torch.set_flush_denormal(True)
torch.manual_seed(0); np.random.seed(0)

### 2. Prototypical networks
An **N-way K-shot episode** gives N new classes with K labeled examples each (the support set) and asks the model to classify more examples of those classes (the query set). A prototypical network (Snell et al. 2017) embeds every image, averages the support embeddings of each class into a **prototype**, and classifies a query by its squared distance to the prototypes. It is trained on episodes built from training classes and tested on episodes from classes it never saw.

Our main dataset is a synthetic alphabet of 100 'glyphs': each class is a random 5x5 binary pattern, and each example puts it at a random position on a 12x12 canvas with noise. Classes 0–79 are for training, 80–99 for testing. The random position makes raw pixel distances almost useless, so a good embedding must learn something that transfers to new glyphs.

In [ ]:
rng = np.random.default_rng(0)
N_GLYPHS, PER_CLASS, CANVAS = 100, 20, 12
patterns = (rng.random((N_GLYPHS, 5, 5)) < 0.4).astype(np.float32)
glyphs = np.zeros((N_GLYPHS, PER_CLASS, CANVAS, CANVAS), np.float32)
for c in range(N_GLYPHS):
    for i in range(PER_CLASS):
        ox, oy = rng.integers(0, CANVAS - 4, 2)                     # random position
        glyphs[c, i, oy:oy + 5, ox:ox + 5] = patterns[c]
glyphs += rng.normal(0, 0.2, glyphs.shape).astype(np.float32)
glyph_data = {c: torch.tensor(glyphs[c]).unsqueeze(1) for c in range(N_GLYPHS)}    # (20, 1, 12, 12) per class

fig, ax = plt.subplots(3, 8, figsize=(8, 3.2))
for r, c in enumerate([0, 1, 2]):
    for j in range(8):
        ax[r, j].imshow(glyph_data[c][j, 0], cmap="gray"); ax[r, j].axis("off")
    ax[r, 0].set_title(f"class {c}", fontsize=8, loc="left")
plt.suptitle("Synthetic glyphs: each row is one class at different positions", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
def episode(data, classes, n_way, k_shot, n_query, g):
    chosen = [classes[i] for i in torch.randperm(len(classes), generator=g)[:n_way]]
    support, query = [], []
    for c in chosen:
        idx = torch.randperm(len(data[c]), generator=g)[:k_shot + n_query]
        support.append(data[c][idx[:k_shot]]); query.append(data[c][idx[k_shot:]])
    return torch.stack(support), torch.stack(query)            # (n_way, k, 1, H, W), (n_way, q, 1, H, W)


def proto_logits(embed, support, query):
    n_way, k = support.shape[:2]
    prototypes = embed(support.flatten(0, 1)).reshape(n_way, k, -1).mean(1)
    return -torch.cdist(embed(query.flatten(0, 1)), prototypes) ** 2      # closer prototype -> larger logit


def few_shot_accuracy(embed, data, classes, k_shot, n_way=5, episodes=400):
    g = torch.Generator().manual_seed(1)
    accs = []
    with torch.no_grad():
        for _ in range(episodes):
            s, q = episode(data, classes, n_way, k_shot, 10, g)
            accs.append((proto_logits(embed, s, q).argmax(1) == torch.arange(n_way).repeat_interleave(10)).float().mean().item())
    return float(np.mean(accs))


def train_protonet(data, classes, n_way=5, episodes=600, seed=0):
    torch.manual_seed(seed)
    embed = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
                          nn.AdaptiveMaxPool2d(2), nn.Flatten(), nn.Linear(128, 64))
    opt = torch.optim.Adam(embed.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for it in range(episodes):
        s, q = episode(data, classes, n_way, 5, 10, g)
        loss = F.cross_entropy(proto_logits(embed, s, q), torch.arange(n_way).repeat_interleave(10))
        opt.zero_grad(); loss.backward(); opt.step()
    return embed


def report(name, data, train_classes, test_classes):
    pixels = nn.Flatten()
    torch.manual_seed(0)
    untrained = train_protonet(data, train_classes, episodes=0)
    trained = train_protonet(data, train_classes)
    print(f"{name}: 5-way accuracy on held-out classes (chance 0.20)")
    for label, emb in [("raw pixels", pixels), ("untrained CNN", untrained), ("prototypical network", trained)]:
        print(f"   {label:21s} 1-shot {few_shot_accuracy(emb, data, test_classes, 1):.3f}   "
              f"5-shot {few_shot_accuracy(emb, data, test_classes, 5):.3f}")

t0 = time.time()
report("glyphs, 80 training classes", glyph_data, list(range(80)), list(range(80, 100)))
print(f"({time.time() - t0:.0f} s)")

Now the same method on scikit-learn's digits, with only **5 training classes** (0–4) and the other 5 (5–9) held out. Every training episode then contains the same five classes.

In [ ]:
digits = load_digits()
Xd = torch.tensor(digits.data, dtype=torch.float32)
Xd = ((Xd - Xd.mean(0)) / (Xd.std(0) + 1e-3)).view(-1, 1, 8, 8)
digit_data = {c: Xd[torch.tensor(digits.target) == c] for c in range(10)}
report("digits, 5 training classes", digit_data, [0, 1, 2, 3, 4], [5, 6, 7, 8, 9])

### 3. First-order MAML on sine waves
MAML (Finn et al. 2017) learns an **initialization** θ such that a few gradient steps on a new task's small support set give a good model. Tasks are sine waves y = A sin(x − φ) with amplitude A in [0.1, 5] and phase φ in [0, π], as in the paper; each task gives 10 support points. The network has two hidden layers of 40 units.

Full MAML backpropagates through the inner gradient steps. **First-order MAML** skips that: it adapts a copy of θ with 5 SGD steps on the support points, computes the gradient of the loss on new query points at the adapted weights, and applies that gradient to θ. Reptile (Nichol et al. 2018) is even simpler: it moves θ towards the adapted weights. The baseline is a network **pre-trained** on all tasks mixed together, then fine-tuned the same way.

In [ ]:
def sine_tasks(n_tasks, n_points, g):
    A = torch.rand(n_tasks, 1, 1, generator=g) * 4.9 + 0.1
    phase = torch.rand(n_tasks, 1, 1, generator=g) * math.pi
    x = torch.rand(n_tasks, n_points, 1, generator=g) * 10 - 5
    return x, A * torch.sin(x - phase), A, phase


def init_params(seed):
    torch.manual_seed(seed)
    layers = [nn.Linear(1, 40), nn.Linear(40, 40), nn.Linear(40, 1)]
    return [t.detach().clone().requires_grad_()
            for m in layers for t in (m.weight.T.contiguous(), m.bias.view(1, -1))]


def net(params, x):
    """x: (tasks, points, 1). params may carry a leading task dimension, so many tasks run at once."""
    W1, b1, W2, b2, W3, b3 = params
    h = torch.relu(x @ W1 + b1)
    h = torch.relu(h @ W2 + b2)
    return h @ W3 + b3


def adapt(params, x, y, steps, alpha=0.01):
    """`steps` plain SGD steps on each task's support set, for all tasks in parallel."""
    fast = [p.detach().unsqueeze(0).repeat(x.shape[0], *[1] * p.dim()).requires_grad_() for p in params]
    for _ in range(steps):
        loss = ((net(fast, x) - y) ** 2).mean((1, 2)).sum()        # sum of the tasks' losses: gradients stay separate
        grads = torch.autograd.grad(loss, fast)
        fast = [(f - alpha * gr).detach().requires_grad_() for f, gr in zip(fast, grads)]
    return fast

t0 = time.time()
theta = init_params(0)
opt = torch.optim.Adam(theta, lr=1e-3)
g = torch.Generator().manual_seed(0)
for it in range(2000):
    x, y, _, _ = sine_tasks(10, 20, g)                                   # 10 tasks per meta-batch
    fast = adapt(theta, x[:, :10], y[:, :10], steps=5)                   # inner loop on 10 support points
    query_loss = ((net(fast, x[:, 10:]) - y[:, 10:]) ** 2).mean((1, 2)).sum()
    grads = torch.autograd.grad(query_loss, fast)
    for p, gr in zip(theta, grads):
        p.grad = gr.mean(0)                                              # first-order meta-gradient
    opt.step()

pretrained = init_params(0)
opt = torch.optim.Adam(pretrained, lr=1e-3)
g = torch.Generator().manual_seed(0)
for it in range(2000):
    x, y, _, _ = sine_tasks(10, 20, g)
    loss = ((net([p.unsqueeze(0) for p in pretrained], x) - y) ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print(f"meta-training and pre-training took {time.time() - t0:.0f} s")

g = torch.Generator().manual_seed(7)
xs, ys, A, phase = sine_tasks(200, 10, g)                                # 200 new tasks, 10 support points each
x_grid = torch.linspace(-5, 5, 100).view(1, 100, 1).expand(200, -1, -1)
y_grid = A * torch.sin(x_grid - phase)
print("mean squared error on the whole curve after k SGD steps on 10 points (200 new tasks)")
print("                     k=0     k=1     k=5    k=10")
for name, start in [("first-order MAML", theta), ("pre-trained", pretrained), ("random init", init_params(1))]:
    row = []
    for k in [0, 1, 5, 10]:
        adapted = adapt(start, xs, ys, k)                                # adaptation needs gradients
        with torch.no_grad():
            row.append(((net(adapted, x_grid) - y_grid) ** 2).mean().item())
    print(f"{name:17s} " + " ".join(f"{v:7.3f}" for v in row))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for a, (name, start) in zip(ax, [("first-order MAML", theta), ("pre-trained", pretrained)]):
    a.plot(x_grid[0, :, 0], y_grid[0, :, 0], "k", lw=2, label="true function")
    a.plot(xs[0, :, 0], ys[0, :, 0], "k^", label="10 support points")
    for k, style in [(0, ":"), (1, "--"), (10, "-")]:
        adapted = adapt(start, xs[:1], ys[:1], k)
        with torch.no_grad():
            pred = net(adapted, x_grid[:1])
        a.plot(x_grid[0, :, 0], pred[0, :, 0], style, label=f"after {k} step{'' if k == 1 else 's'}")
    a.set(title=name, xlabel="x"); a.legend(fontsize=7)
ax[0].set_ylabel("y")
plt.tight_layout(); plt.show()

### 4. In-context learning: a tiny transformer that does regression without changing its weights
Garg et al. (2022) trained transformers on prompts (x₁, y₁, x₂, y₂, …, x_q) where y = w·x and **each prompt has its own random w**. The only way to predict y_q well is to work out w from the examples in the prompt, during the forward pass. We do the same with dimension 4, up to 12 examples, and a 3-layer transformer of width 64. Tokens alternate: an x-token holds (x, 0) and a y-token holds (0, 0, 0, 0, y); a causal mask lets each position see only earlier tokens, and the prediction for y_i is read at the x_i token. One training run sees all prompt lengths at once.

In [ ]:
D, K = 4, 12

def regression_prompts(n, g):
    w = torch.randn(n, D, generator=g)
    x = torch.randn(n, K, D, generator=g)
    return x, (x @ w[:, :, None]).squeeze(-1)                          # noiseless y = w.x, a new w per prompt

def to_tokens(x, y):
    n = len(x)
    x_tok = torch.cat([x, torch.zeros(n, K, 1)], -1)
    y_tok = torch.cat([torch.zeros(n, K, D), y[..., None]], -1)
    return torch.stack([x_tok, y_tok], 2).reshape(n, 2 * K, D + 1)     # x1, y1, x2, y2, ...


class CausalBlock(nn.Module):
    def __init__(self, d, heads, mlp_ratio=2):
        super().__init__()
        self.heads = heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, mlp_ratio * d), nn.GELU(), nn.Linear(mlp_ratio * d, d))

    def forward(self, h):
        B, T, Dm = h.shape
        q, k, v = self.qkv(self.ln1(h)).view(B, T, 3, self.heads, Dm // self.heads).permute(2, 0, 3, 1, 4)
        att = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        h = h + self.proj(att.transpose(1, 2).reshape(B, T, Dm))
        return h + self.mlp(self.ln2(h))


class InContextRegressor(nn.Module):
    def __init__(self, d=64, depth=3, heads=4):
        super().__init__()
        self.read_in = nn.Linear(D + 1, d)
        self.pos = nn.Parameter(torch.randn(1, 2 * K, d) * 0.02)
        self.blocks = nn.ModuleList([CausalBlock(d, heads) for _ in range(depth)])
        self.norm, self.read_out = nn.LayerNorm(d), nn.Linear(d, 1)

    def forward(self, x, y):
        h = self.read_in(to_tokens(x, y)) + self.pos
        for block in self.blocks:
            h = block(h)
        return self.read_out(self.norm(h))[:, 0::2, 0]                  # prediction for y_i, read at token x_i

t0 = time.time()
torch.manual_seed(0)
icl = InContextRegressor()
opt = torch.optim.AdamW(icl.parameters(), lr=2e-3, weight_decay=0.0)
g = torch.Generator().manual_seed(0)
STEPS = 2000
for step in range(STEPS):
    for group in opt.param_groups:
        group["lr"] = 2e-3 * min(1.0, (step + 1) / 200) * 0.5 * (1 + math.cos(math.pi * step / STEPS))
    x, y = regression_prompts(64, g)
    loss = F.mse_loss(icl(x, y), y)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"trained for {STEPS} steps in {time.time() - t0:.0f} s; parameters: {sum(p.numel() for p in icl.parameters()):,}")

Now we freeze the transformer and compare it with three ways of predicting y_q from k examples:
- **zero**: always predict 0 (normalized error 1);
- **one step of gradient descent** from w = 0 on the k examples, ŵ = η Σ yᵢxᵢ, with the best step size η for each k (fitted on separate prompts);
- **least squares**: the minimum-norm solution, which is exact once k ≥ 4 because the data have no noise.

Errors are squared errors divided by D = 4, the variance of y, so 1.0 means 'no better than predicting 0'.

In [ ]:
def baselines(x, y):
    n = len(x)
    ols, gd_feature = torch.zeros(n, K), torch.zeros(n, K)
    for k in range(1, K):
        Xk, yk = x[:, :k], y[:, :k]
        w_ols = torch.linalg.pinv(Xk) @ yk[..., None]                     # minimum-norm least squares
        ols[:, k] = (w_ols[..., 0] * x[:, k]).sum(-1)
        gd_feature[:, k] = ((yk[..., None] * Xk).sum(1) * x[:, k]).sum(-1)   # (sum_i y_i x_i) . x_query
    return ols, gd_feature

g_fit, g_test = torch.Generator().manual_seed(11), torch.Generator().manual_seed(12)
x_fit, y_fit = regression_prompts(4000, g_fit)
_, a_fit = baselines(x_fit, y_fit)
eta = (a_fit * y_fit).sum(0) / (a_fit ** 2).sum(0).clamp_min(1e-9)     # best step size for each k

x_te, y_te = regression_prompts(4000, g_test)
with torch.no_grad():
    pred_icl = icl(x_te, y_te)
ols_te, a_te = baselines(x_te, y_te)
err = lambda pred: (((pred - y_te) ** 2).mean(0) / D).numpy()
curves = {"zero": err(torch.zeros_like(y_te)), "one GD step (best eta)": err(eta * a_te),
          "least squares": err(ols_te), "transformer, in context": err(pred_icl)}
print("normalized error by number of examples in the prompt k")
print("k:                        " + " ".join(f"{k:6d}" for k in range(K)))
for name, c in curves.items():
    print(f"{name:25s} " + " ".join(f"{v:6.3f}" for v in c))

plt.figure(figsize=(6.5, 3.5))
for name, c in curves.items():
    plt.plot(range(K), c, "o-", ms=4, label=name)
plt.axvline(D, color="gray", ls=":", lw=1)
plt.title("In-context linear regression, dimension 4 (dotted: k = 4)")
plt.xlabel("number of examples in the prompt, k"); plt.ylabel("normalized squared error")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

### 5. Induction heads in a real language model (NOT RUN here)
Olsson et al. (2022) argue that much in-context learning in language models runs through **induction heads**, attention heads that find an earlier occurrence of the current token and copy what followed it. A simple test: feed a random token sequence twice. A model with induction heads predicts the second copy far better than the first. The cell uses the documented Hugging Face `transformers` interface.

In [ ]:
# NOT RUN: needs `pip install transformers` and a download of GPT-2 small (124M parameters); runs on a CPU in seconds.
import torch
from transformers import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
torch.manual_seed(0)
seq = torch.randint(1000, 10000, (1, 50))                  # 50 random tokens
ids = torch.cat([seq, seq], dim=1)                         # ... repeated once
with torch.no_grad():
    logits = model(ids).logits
loss = torch.nn.functional.cross_entropy(logits[0, :-1], ids[0, 1:], reduction="none")
print(f"mean loss on the first copy:  {loss[:49].mean():.2f} nats")
print(f"mean loss on the second copy: {loss[50:].mean():.2f} nats (much lower: the model copies from context)")

### TODO
1. Train the transformer of section 4 for 500 steps instead of 2,000. At which k does it fall behind least squares the most?
2. Test the trained transformer on prompts whose x has twice the standard deviation (a distribution shift). Does it still track least squares?
3. Complete Reptile below and compare it with first-order MAML after 1 and 10 adaptation steps.

In [ ]:
# TODO: one Reptile meta-update (Nichol et al. 2018). Adapt to each task with `adapt` (5 steps), then move theta a
# fraction epsilon of the way towards the average of the adapted weights: theta <- theta + epsilon * (mean(fast) - theta).
def reptile_step(params, x, y, epsilon=0.1):
    return None          # replace: fast = adapt(params, x, y, steps=5); update each p in place (inside torch.no_grad())

result = reptile_step(init_params(0), *sine_tasks(10, 10, torch.Generator().manual_seed(0))[:2])
print("not done yet" if result is None else "Reptile step done")

### What to notice
- **Prototypical networks need many training classes.** On the synthetic glyphs, with 80 training classes, the prototypical network reached 0.586 (1-shot) and 0.730 (5-shot) on 20 unseen classes, against 0.216 and 0.234 for raw pixels and 0.268 and 0.309 for the same CNN untrained. On digits with only 5 training classes it went the other way: 0.532 and 0.694, worse than raw pixels (0.656 and 0.848) and even worse than the untrained CNN (0.564 and 0.716). With five classes every training episode is the same task, so the embedding specializes in digits 0–4 instead of learning what separates classes in general. While preparing this lab we also tried an MLP embedding, random shifts and 7 training classes; none beat raw pixels on held-out digits. This is why benchmarks such as Omniglot have more than a thousand classes, and it is the same lesson about task diversity as in Raventós et al. (2023).
- **MAML learns to adapt, not to predict.** Before any adaptation, the MAML initialization fitted new sine waves worse than the pre-trained network (MSE 3.450 against 2.931). After one SGD step on 10 points it was much better (1.473 against 2.428), and after 10 steps it reached 0.595 against 1.520. The pre-trained network learned an average curve; MAML learned a starting point from which a single step goes a long way.
- **In-context learning.** With its weights frozen, the transformer's error fell from 0.992 with no examples to 0.186 at k = 4 and 0.026 at k = 11. It tracked least squares closely while k < 4 (0.726, 0.510 and 0.323, against 0.722, 0.489 and 0.266), and it was far better than one step of gradient descent with the best step size (0.295 at k = 11). Unlike least squares it did not become exact at k = 4 (0.186 against 0.000). This small model behaves like an approximate least-squares solver, not like a single gradient step; Garg et al. (2022) and Akyürek et al. (2023) report similar matches at larger scale.
- **What this does not show.** The transformer was trained on this exact family of tasks; it says nothing direct about how language models learn in context from text. We also did not look inside the model, so we cannot say which algorithm it runs.
- **Limits.** One seed for every experiment.